# Procurement Cycle-Time Analysis

## Project Overview
Analyze procurement workflow timestamps to measure approval delays, vendor delays, and total purchase cycle time. This is a descriptive analytics project.

## Learning Objectives
- Calculate cycle times across procurement stages
- Identify bottlenecks in the approval and delivery pipeline
- Compare cycle times by category, vendor, and urgency
- Detect trends and outliers in procurement speed

## Problem Statement
Procurement takes too long. Management needs data on where time is spent — internal approvals, vendor response, or delivery — to target process improvements.

## Why This Project Matters
Long procurement cycles delay projects, increase costs, and frustrate stakeholders. Pinpointing bottlenecks enables targeted process optimization.

## Dataset Overview
Synthetic procurement records: ~3,000 purchase orders over 2 years with timestamps for key workflow stages.

## Dataset Source and License Notes
- Synthetic data for educational purposes
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n = 3000
categories = np.random.choice(['IT Equipment', 'Office Supplies', 'Raw Materials', 'Services', 'MRO'], n,
                                p=[0.20, 0.25, 0.15, 0.20, 0.20])
urgency = np.random.choice(['Standard', 'Urgent', 'Critical'], n, p=[0.60, 0.30, 0.10])
vendors = np.random.choice([f'Vendor-{i:02d}' for i in range(1, 21)], n)
departments = np.random.choice(['Engineering', 'Operations', 'Marketing', 'Finance', 'HR'], n,
                                p=[0.25, 0.30, 0.15, 0.15, 0.15])

base_dates = pd.date_range('2023-01-01', '2024-12-31', freq='D')
request_dates = np.array([pd.Timestamp(d) for d in np.random.choice(base_dates, n)])

# Stage durations (days)
urgency_mult = {'Standard': 1.0, 'Urgent': 0.6, 'Critical': 0.3}
approval_days = np.array([max(1, int(np.random.exponential(5) * urgency_mult[u])) for u in urgency])
vendor_response_days = np.array([max(1, int(np.random.exponential(3))) for _ in range(n)])
delivery_days = np.array([max(1, int(np.random.exponential(8))) for _ in range(n)])
receipt_inspect_days = np.array([max(1, int(np.random.exponential(2))) for _ in range(n)])

approval_date = request_dates + pd.to_timedelta(approval_days, unit='D')
po_sent_date = approval_date + pd.to_timedelta(vendor_response_days, unit='D')
delivery_date = po_sent_date + pd.to_timedelta(delivery_days, unit='D')
complete_date = delivery_date + pd.to_timedelta(receipt_inspect_days, unit='D')

total_cycle = ((complete_date - request_dates) / pd.Timedelta(days=1)).astype(int)

df = pd.DataFrame({
    'POID': [f'PO{i:06d}' for i in range(n)],
    'RequestDate': request_dates, 'ApprovalDate': approval_date,
    'POSentDate': po_sent_date, 'DeliveryDate': delivery_date,
    'CompleteDate': complete_date,
    'Category': categories, 'Urgency': urgency,
    'Vendor': vendors, 'Department': departments,
    'ApprovalDays': approval_days, 'VendorResponseDays': vendor_response_days,
    'DeliveryDays': delivery_days, 'ReceiptDays': receipt_inspect_days,
    'TotalCycleDays': total_cycle,
})
df['YearMonth'] = df['RequestDate'].dt.to_period('M')

print(f'Dataset shape: {df.shape}')
print(f'Avg total cycle: {df["TotalCycleDays"].mean():.1f} days')
print(f'Median total cycle: {df["TotalCycleDays"].median():.0f} days')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Date range: {df["RequestDate"].min().date()} to {df["RequestDate"].max().date()}')
print(f'\nUrgency distribution:\n{df["Urgency"].value_counts()}')
print(f'\nCategory distribution:\n{df["Category"].value_counts()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df['TotalCycleDays'].hist(bins=50, ax=axes[0,0], edgecolor='black', alpha=0.7, color='steelblue')
axes[0,0].set_title('Total Cycle Time Distribution')
axes[0,0].axvline(df['TotalCycleDays'].median(), color='red', linestyle='--', label=f'Median: {df["TotalCycleDays"].median():.0f}d')
axes[0,0].legend()

stages = ['ApprovalDays', 'VendorResponseDays', 'DeliveryDays', 'ReceiptDays']
stage_means = df[stages].mean()
stage_means.plot.bar(ax=axes[0,1], edgecolor='black', color=['coral','steelblue','green','purple'])
axes[0,1].set_title('Avg Duration by Stage')
axes[0,1].set_ylabel('Days')
axes[0,1].tick_params(axis='x', rotation=45)

df.groupby('Urgency')['TotalCycleDays'].mean().sort_values().plot.barh(ax=axes[1,0], edgecolor='black', color='teal')
axes[1,0].set_title('Avg Cycle Time by Urgency')

monthly_cycle = df.groupby('YearMonth')['TotalCycleDays'].mean()
monthly_cycle.plot(ax=axes[1,1], marker='o', color='coral')
axes[1,1].set_title('Monthly Avg Cycle Time Trend')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Stage Breakdown by Category

In [ ]:
cat_stages = df.groupby('Category')[stages].mean().round(1)
print('Avg Stage Duration by Category (days):')
print(cat_stages)

fig, ax = plt.subplots(figsize=(12, 6))
cat_stages.plot.bar(stacked=True, ax=ax, edgecolor='black')
ax.set_title('Cycle Time Composition by Category')
ax.set_ylabel('Days')
ax.legend(title='Stage', bbox_to_anchor=(1.05, 1))
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'category_stages.png'), dpi=100, bbox_inches='tight')
plt.show()

## Department Comparison

In [ ]:
dept = df.groupby('Department').agg(
    orders=('POID', 'count'),
    avg_cycle=('TotalCycleDays', 'mean'),
    median_cycle=('TotalCycleDays', 'median'),
    avg_approval=('ApprovalDays', 'mean'),
).round(1).sort_values('avg_cycle', ascending=False)
print('Department Cycle Times:')
print(dept)

fig, ax = plt.subplots(figsize=(10, 5))
dept['avg_cycle'].plot.barh(ax=ax, edgecolor='black', color='mediumpurple')
ax.set_title('Avg Total Cycle by Department')
ax.set_xlabel('Days')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'department.png'), dpi=100, bbox_inches='tight')
plt.show()

## Vendor Performance

In [ ]:
vendor_stats = df.groupby('Vendor').agg(
    orders=('POID', 'count'),
    avg_delivery=('DeliveryDays', 'mean'),
    avg_response=('VendorResponseDays', 'mean'),
).round(1).sort_values('avg_delivery', ascending=False)
print('Top 10 Slowest Vendors by Delivery:')
print(vendor_stats.head(10))

fig, ax = plt.subplots(figsize=(12, 6))
vendor_stats.head(15)['avg_delivery'].plot.barh(ax=ax, edgecolor='black', color='coral')
ax.set_title('Avg Delivery Days — Top 15 Slowest Vendors')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'vendor_performance.png'), dpi=100, bbox_inches='tight')
plt.show()

## Cycle Time by Urgency × Category

In [ ]:
cross = df.groupby(['Urgency', 'Category'])['TotalCycleDays'].mean().unstack().round(1)
print('Avg Cycle — Urgency × Category:')
print(cross)

fig, ax = plt.subplots(figsize=(10, 5))
cross.plot.bar(ax=ax, edgecolor='black')
ax.set_title('Avg Cycle Time: Urgency × Category')
ax.set_ylabel('Days')
ax.legend(title='Category', bbox_to_anchor=(1.05, 1))
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'urgency_category.png'), dpi=100, bbox_inches='tight')
plt.show()

## Outlier Detection — Long Cycle Orders

In [ ]:
p95 = df['TotalCycleDays'].quantile(0.95)
outliers = df[df['TotalCycleDays'] > p95]
print(f'P95 cycle time: {p95:.0f} days')
print(f'Orders above P95: {len(outliers)} ({len(outliers)/len(df)*100:.1f}%)')
print(f'\nOutlier breakdown by category:\n{outliers["Category"].value_counts()}')
print(f'\nOutlier avg stage durations:')
print(outliers[stages].mean().round(1))

## Interpretation and Business Insight
- **Delivery** is the longest stage on average, followed by **Approval** — these are the primary optimization targets
- **Critical** urgency orders have significantly faster approval but similar delivery times — vendor SLAs may need tightening
- **Department** differences in approval time suggest varying internal process maturity
- **Vendor performance** varies widely — consolidating to faster vendors or setting delivery SLAs would help
- **Outliers** (P95+) are disproportionately caused by slow deliveries, not approvals — supply chain focus needed

## Limitations
- Synthetic data with simplified stage logic
- No cost data to quantify the financial impact of delays
- No approval hierarchy complexity (single vs multi-level)
- No rework or rejection loops modeled
- No seasonal vendor capacity constraints

## How to Improve This Project
- Use real ERP procurement data (SAP, Oracle)
- Add cost-of-delay analysis (project delays, production hold costs)
- Include approval routing complexity (multi-level, delegation)
- Build predictive models for cycle time estimation
- Add SLA compliance tracking per vendor

## Production Considerations
- Real-time procurement dashboards with stage-level tracking
- Automated escalation when any stage exceeds SLA threshold
- Vendor scorecards integrated with procurement cycle data
- Monthly process improvement reviews with stage breakdown

## Common Mistakes
- Measuring only total cycle time without stage decomposition
- Not accounting for urgency when benchmarking departments
- Ignoring outliers that skew averages significantly
- Treating all procurement categories with the same time expectations

## Mini Challenge / Exercises
1. What % of Standard orders complete within 20 days? Within 30 days?
2. Which stage has the highest coefficient of variation (most unpredictable)?
3. If approval times were halved for Standard orders, what would the new average total cycle be?
4. Which vendor + category combination produces the longest average delivery time?

## Final Summary / Key Takeaways
- Procurement cycle-time analysis decomposes delays into actionable stages
- Delivery and approval are the highest-impact optimization targets
- Urgency classification significantly accelerates approvals but less so deliveries
- Vendor and department comparisons reveal systemic differences worth addressing
- Stage-level visibility is essential for continuous process improvement